In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns

# Hypothesis testing: Chi-Square Test within the Eniac case study

In this notebook we perform a chi-square test with the data from the Eniac case study and use the click-through rates to find the winning version.

## 1.&nbsp;State the Null Hypothesis and the Alternative Hypothesis.

Null Hypothesis ( H0 ): The click through rate for all versions of the website is equal.

Alternative Hypothesis ( HA ): The click-through rate for at least one version of the website differs.

## 2.&nbsp; Select an appropriate significance level alpha ($\alpha$).

It was decided that a relatively high alpha was acceptable in this case

In [2]:
alpha = 0.05

## 3.&nbsp; Collect data that is random and independent

The important pieces of information (clicks on each element of interest & visits on each page) are scattered around. Let's collect them. Where are the .csv files? 🥸

In [3]:
from google.colab import files
uploaded = files.upload()  # opens a file picker, select all your CSVs at once

Saving eniac_a.csv to eniac_a.csv
Saving eniac_b.csv to eniac_b.csv
Saving eniac_c.csv to eniac_c.csv
Saving eniac_d.csv to eniac_d.csv


In [4]:
eniac_a_df = pd.read_csv('eniac_a.csv')   # filename in quotes (it's a string)
eniac_b_df = pd.read_csv('eniac_b.csv')
eniac_c_df = pd.read_csv('eniac_c.csv')
eniac_d_df = pd.read_csv('eniac_d.csv')
eniac_a_df.head()                        # now you're looking at the actual variable

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,269,True,Homepage Version A - white SHOP NOW • http...
1,25,div,mySidebar,309,True,created 2021-09-14 • 14 days 0 hours 34 mi...
2,4,a,Mac,279,True,NaN
3,69,a,iPhone,246,True,NaN
4,105,a,Accessories,1235,True,NaN


In [5]:
#finding total no.of clicks in each test.
eniac_a_clicks = eniac_a_df.loc[eniac_a_df["Name"]=="SHOP NOW", "No. clicks"].iloc[0]
eniac_b_clicks = eniac_b_df.loc[eniac_b_df["Name"]=="SHOP NOW", "No. clicks"].iloc[0]
eniac_c_clicks = eniac_c_df.loc[eniac_c_df["Name"]=="SEE DEALS", "No. clicks"].iloc[0]
eniac_d_clicks = eniac_d_df.loc[eniac_d_df["Name"]=="SEE DEALS", "No. clicks"].iloc[0]

In [6]:
eniac_a_clicks = eniac_a_df.loc[eniac_a_df["Name"]=="SHOP NOW", "No. clicks"].iloc[0]
eniac_a_clicks

np.int64(512)

In [7]:
#finding total no.of visits in each test.
eniac_a_df.iloc[1, -1]

'created 2021-09-14   •   14 days 0 hours 34 mins   •   25326 visits, 23174 clicks'

In [8]:
#finding total no.of visits in each test.
eniac_b_df.iloc[1, -1]

'created 2021-10-27   •   14 days 0 hours 34 mins   •   24747 visits, 22407 clicks'

In [9]:
#finding total no.of visits in each test.
eniac_c_df.iloc[1, -1]

'created 2021-10-27   •   14 days 0 hours 34 mins   •   24876 visits, 23031 clicks'

In [10]:
#finding total no.of visits in each test.
eniac_d_df.iloc[1, -1]

'created 2021-10-27   •   14 days 0 hours 34 mins   •   25233 visits, 22743 clicks'

In [11]:
# added all visits manually based on the above code
eniac_a_visits = 25326
eniac_b_visits = 24747
eniac_c_visits = 24876
eniac_d_visits = 25233

In [12]:
visits = [eniac_a_visits, eniac_b_visits, eniac_c_visits, eniac_d_visits]
clicks = [eniac_a_clicks, eniac_b_clicks, eniac_c_clicks, eniac_d_clicks]

observed_results = pd.DataFrame(data = {"Visits": visits, "Click": clicks},
                                index = ["Version_A", "Version_B", "Version_C", "Version_D"]
                                )
observed_results["No-click"]=observed_results["Visits"]-observed_results["Click"]

observed_results

,Visits,Click,No-click
Version_A,25326,512,24814
Version_B,24747,281,24466
Version_C,24876,527,24349
Version_D,25233,193,25040


## 4.&nbsp; Calculate the test result

Calculating degrees of freedom
dof=(r−1)×(n−1)

In [13]:
r=observed_results.shape[0]
r

4

In [14]:
n=observed_results.shape[1]
n

3

In [15]:
dof = (r-1) * (n-1)
dof

6

Calculating the expected value for each cell

In [16]:
# Calculate row totals - stored as a vertical DataFrame
row_totals = observed_results.sum(axis=1).to_frame()

# Calculate column totals - stored as a horizontal DataFrame
column_totals = observed_results.sum(axis=0).to_frame().T

# Calculate the grand total - a single value
grand_total = observed_results.values.sum()

# Calculate the population proportions
proportions = column_totals/grand_total

# Multiply proportions by row totals to get expected values
expected_results = row_totals.dot(proportions)

expected_results

,Visits,Click,No-click
Version_A,25326.0,382.486255,24943.513745
Version_B,24747.0,373.741900,24373.258100
Version_C,24876.0,375.690124,24500.309876
Version_D,25233.0,381.081721,24851.918279


Calculating the Chi-squared statistic

In [17]:
chi_squared = (
    ((observed_results - expected_results) ** 2 / expected_results)
    .sum() # add up the columns
    .sum() # add column sums together
)
chi_squared

np.float64(224.0187748805841)

In [18]:
critical_value = 12.592

In [19]:
chisq, pvalue, df, expected = stats.chi2_contingency(observed_results[["Click", "No-click"]])

In [20]:
pvalue

np.float64(2.71612166078691e-48)

## 5.&nbsp; Interpret the test result

In [21]:
if pvalue > alpha:
    print("The p-value is larger than alpha.")
else:
    print("The p-value is smaller than alpha.")

The p-value is smaller than alpha.


## How do we decide who's the winner?

p-value is smaller than alpha, we do reject the null hypothesis.

Since the p-value is smaller than alpha, we will reject the Null Hypothesis. Therefore, the four versions have not performed equally well.

But which version is the best one?

Let's decide to look at the click-through rates of the different versions. They might help us to declare a winner.

In [22]:
# calculate the click through rate for each version
ctr_a = observed_results.loc["Version_A", "Click"] / observed_results.loc["Version_A", "Visits"]
ctr_b = observed_results.loc["Version_B", "Click"] / observed_results.loc["Version_B", "Visits"]
ctr_c = observed_results.loc["Version_C", "Click"] / observed_results.loc["Version_C", "Visits"]
ctr_d = observed_results.loc["Version_D", "Click"] / observed_results.loc["Version_D", "Visits"]

# create a table with all four versions and their click-through rates.
ctrs = pd.DataFrame(
    {"version": ["Version_A", "Version_B", "Version_C", "Version_D"], "CTR": [ctr_a, ctr_b, ctr_c, ctr_d]}
)

ctrs.sort_values("CTR", ascending=False)

,version,CTR
2,Version_C,0.021185
0,Version_A,0.020216
1,Version_B,0.011355
3,Version_D,0.007649


## Conclusion
The chi-square test rejects the null hypothesis, so the four versions do not perform equally.
Based on click-through rate, Version C (white SEE DEALS, 2.12%) and Version A (white SHOP NOW, 2.02%) are the winners.
The red versions perform worst (B: 1.14%, D: 0.76%).